# Phase 03L: Teacher C Aggregate and Phase 3 Complete

Notebook này là cổng cuối của Phase 3.

Nhiệm vụ:

1. xác minh ba full run Teacher C seed 42/123/2026;
2. kiểm toàn bộ manifest, SHA-256, Validation và checkpoint;
3. tổng hợp mean, sample SD và CV của Teacher C;
4. lập checkpoint registry và model card;
5. xác minh Teacher A, B và C đều hoàn tất;
6. khóa compatibility cho ba cặp Teacher–Student;
7. phát hành `PHASE3_TEACHER_POOL_COMPLETE.json`;
8. phát hành `PHASE4_KD_REPRODUCTION_AUTHORIZED.json`.

Held-out Test luôn `SEALED`; notebook không mở checkpoint payload, ảnh hoặc
caption.

## Kaggle inputs

- Phase 03E Teacher A aggregate.
- Phase 03H Teacher B aggregate.
- Phase 03I Teacher C preflight.
- Ba output Phase 03K Teacher C seeds 42, 123 và 2026.

## Settings

- CPU là đủ.
- Internet Off.
- Chạy tuần tự Cell 1 → Cell 7.


## Cell 1 — Discovery và khóa upstream

**Đầu ra dự kiến:** tìm đúng ba Teacher C COMPLETE marker, một Teacher A
aggregate, một Teacher B aggregate và đúng SigLIP source contract.


In [1]:
import csv
import hashlib
import json
import math
import os
import platform
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path


CELL_STARTED_AT = time.perf_counter()
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase3_teacher_c_three_seed_aggregate"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
RESULT_DIR = OUTPUT_ROOT / "results"
MODEL_CARD_DIR = OUTPUT_ROOT / "model_cards"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, RESULT_DIR,
    MODEL_CARD_DIR, MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


teacher_c_markers = sorted(
    INPUT_ROOT.rglob("TEACHER_C_SEED*_FULL_RUN_COMPLETE.json")
)
teacher_a_matches = sorted(
    INPUT_ROOT.rglob("PHASE3_TEACHER_A_THREE_SEED_COMPLETE.json")
)
teacher_b_matches = sorted(
    INPUT_ROOT.rglob(
        "PHASE3_TEACHER_B_THREE_SEED_AGGREGATE_COMPLETE.json"
    )
)
source_contract_matches = sorted(
    INPUT_ROOT.rglob(
        "teacher_c_siglip_pretrained_source_contract.json"
    )
)
assert len(teacher_c_markers) == 3, teacher_c_markers
assert len(teacher_a_matches) == 1, teacher_a_matches
assert len(teacher_b_matches) == 1, teacher_b_matches
assert len(source_contract_matches) == 1, source_contract_matches

marker_headers = [read_json(path) for path in teacher_c_markers]
assert {int(item["seed"]) for item in marker_headers} == {42, 123, 2026}
assert all(item["status"] == "complete" for item in marker_headers)
assert all(item["teacher_id"] == "C" for item in marker_headers)
assert all(item["test_set_status"] == "sealed" for item in marker_headers)
assert not any(bool(item["test_set_used"]) for item in marker_headers)

TEACHER_A_COMPLETE_PATH = teacher_a_matches[0].resolve()
TEACHER_B_COMPLETE_PATH = teacher_b_matches[0].resolve()
SOURCE_CONTRACT_PATH = source_contract_matches[0].resolve()
TEACHER_A_COMPLETE = read_json(TEACHER_A_COMPLETE_PATH)
TEACHER_B_COMPLETE = read_json(TEACHER_B_COMPLETE_PATH)
SOURCE_CONTRACT = read_json(SOURCE_CONTRACT_PATH)
assert TEACHER_A_COMPLETE["status"] == "complete"
assert TEACHER_A_COMPLETE["teacher_id"] == "A"
assert TEACHER_A_COMPLETE["qualified_for_kd"] is True
assert TEACHER_B_COMPLETE["status"] == "complete"
assert TEACHER_B_COMPLETE["teacher_id"] == "B"
assert TEACHER_B_COMPLETE["all_qualified_for_kd"] is True
assert SOURCE_CONTRACT["status"] == "locked"
assert SOURCE_CONTRACT["model_identifier"] == (
    "google/siglip-base-patch16-224"
)
assert SOURCE_CONTRACT["resolved_commit"] == (
    "7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed"
)

write_json(
    AUDIT_DIR / "phase3l_cell01_discovery.json",
    {
        "audit_version": "phase3l_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "teacher_c_markers": [str(path) for path in teacher_c_markers],
        "teacher_a_complete": str(TEACHER_A_COMPLETE_PATH),
        "teacher_b_complete": str(TEACHER_B_COMPLETE_PATH),
        "siglip_revision": SOURCE_CONTRACT["resolved_commit"],
        "dataset_read": False,
        "caption_read": False,
        "checkpoint_payload_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03L CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher C COMPLETE runs   : 3 / 3")
print("Teacher A aggregate       : VERIFIED")
print("Teacher B aggregate       : VERIFIED")
print("SigLIP revision           :", SOURCE_CONTRACT["resolved_commit"])
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Next                      : verify run artifacts")


PHASE 03L CELL 1 STATUS
Status                    : PASSED
Teacher C COMPLETE runs   : 3 / 3
Teacher A aggregate       : VERIFIED
Teacher B aggregate       : VERIFIED
SigLIP revision           : 7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed
Checkpoint payload read   : False
Test Set status           : SEALED
Next                      : verify run artifacts


## Cell 2 — Manifest, checkpoint và Validation integrity

**Đầu ra dự kiến:** 95 manifest rows, sáu checkpoint, 60 Validation artifacts
và 21 audit file đều hợp lệ.


In [2]:
CELL_STARTED_AT = time.perf_counter()
run_records = []
manifest_rows_total = 0
sha_rows_total = 0
checkpoint_count = 0
validation_file_count = 0
audit_file_count = 0

for marker_path in teacher_c_markers:
    marker = read_json(marker_path)
    seed = int(marker["seed"])
    run_root = marker_path.resolve().parent
    assert marker["teacher"] == (
        "SigLIP ViT-B/16 + shared caption decoder"
    )
    assert int(marker["completed_epochs"]) == 20
    assert int(marker["global_optimizer_steps"]) == 35420
    assert marker["selection_split"] == "validation"
    assert marker["quality_gate_passed"] is True
    assert marker["qualified_for_kd"] is True
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

    manifest_csv = resolve_inside(
        run_root, marker["manifests"]["csv"]["relative_path"]
    )
    manifest_json = resolve_inside(
        run_root, marker["manifests"]["json"]["relative_path"]
    )
    sha_file = resolve_inside(
        run_root, marker["manifests"]["sha256sums"]["relative_path"]
    )
    for key, path in [
        ("csv", manifest_csv),
        ("json", manifest_json),
        ("sha256sums", sha_file),
    ]:
        assert path.is_file()
        assert sha256_file(path) == marker["manifests"][key]["sha256"]

    with manifest_csv.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    manifest_payload = read_json(manifest_json)
    assert int(manifest_payload["artifact_count"]) == len(rows)
    assert len(manifest_payload["artifacts"]) == len(rows)
    expected_sha_lines = []
    for row in rows:
        artifact = resolve_inside(run_root, row["relative_path"])
        assert artifact.is_file(), artifact
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"]
        expected_sha_lines.append(
            f'{row["sha256"]}  {row["relative_path"]}'
        )
    actual_sha_lines = [
        line.strip()
        for line in sha_file.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    assert actual_sha_lines == expected_sha_lines

    for kind in ["best_checkpoint", "last_checkpoint"]:
        checkpoint = resolve_inside(
            run_root, marker[kind]["relative_path"]
        )
        assert checkpoint.is_file()
        assert sha256_file(checkpoint) == marker[kind]["sha256"]
        checkpoint_count += 1

    validation_files = sorted(
        (run_root / "validation").glob("epoch_*_validation.json")
    )
    assert len(validation_files) == 20
    validation_file_count += len(validation_files)
    audit_files = sorted((run_root / "audits").glob("*.json"))
    assert len(audit_files) == 7
    assert all(read_json(path)["status"] == "passed" for path in audit_files)
    audit_file_count += len(audit_files)

    training_audit_path = (
        run_root / "audits" / "phase3k_cell07_full_training.json"
    )
    training_audit = read_json(training_audit_path)
    assert training_audit["best_result"] == marker["best_result"]
    assert training_audit["quality_gate_passed"] is True

    manifest_rows_total += len(rows)
    sha_rows_total += len(actual_sha_lines)
    run_records.append(
        {
            "seed": seed,
            "run_root": str(run_root),
            "marker_path": str(marker_path.resolve()),
            "marker_sha256": sha256_file(marker_path),
            "marker": marker,
            "manifest_rows": len(rows),
            "training_audit": training_audit,
        }
    )

assert sorted(item["seed"] for item in run_records) == [42, 123, 2026]
assert manifest_rows_total == 95
assert sha_rows_total == 95
assert checkpoint_count == 6
assert validation_file_count == 60
assert audit_file_count == 21

write_json(
    AUDIT_DIR / "phase3l_cell02_integrity.json",
    {
        "audit_version": "phase3l_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "manifest_rows_verified": manifest_rows_total,
        "sha256sum_rows_verified": sha_rows_total,
        "best_last_checkpoints_verified": checkpoint_count,
        "validation_artifacts_verified": validation_file_count,
        "audit_files_verified": audit_file_count,
        "checkpoint_payload_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03L CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Manifest/SHA rows         : 95 / 95 VERIFIED")
print("Best/last checkpoints     : 6 / 6 VERIFIED")
print("Validation artifacts      : 60 / 60 VERIFIED")
print("Run audits                : 21 / 21 PASSED")
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Next                      : audit metric protocol and KD compatibility")


PHASE 03L CELL 2 STATUS
Status                    : PASSED
Manifest/SHA rows         : 95 / 95 VERIFIED
Best/last checkpoints     : 6 / 6 VERIFIED
Validation artifacts      : 60 / 60 VERIFIED
Run audits                : 21 / 21 PASSED
Checkpoint payload read   : False
Test Set status           : SEALED
Next                      : audit metric protocol and KD compatibility


## Cell 3 — Metric protocol và KD compatibility

**Đầu ra dự kiến:** mọi metric hữu hạn; cả ba quality gate đạt; logit,
feature và sequence KD được khóa với điều kiện tương thích rõ ràng.


In [3]:
CELL_STARTED_AT = time.perf_counter()
metric_names = ["BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D"]
overflow_by_seed = {}
training_hours_by_seed = {}
for run in run_records:
    marker = run["marker"]
    best = marker["best_result"]
    assert int(best["epoch"]) in range(1, 21)
    assert int(best["prediction_count"]) == 5000
    assert best["test_set_status"] == "sealed"
    assert best["test_set_used"] is False
    for metric_name in metric_names:
        assert math.isfinite(float(best["metrics"][metric_name]))
    assert math.isfinite(float(best["teacher_forced_ce"]))
    quality_margin = (
        float(best["metrics"]["CIDEr-D"])
        - float(marker["quality_threshold"])
    )
    assert quality_margin >= 0.0
    overflow_by_seed[run["seed"]] = int(
        run["training_audit"]["overflow_retry_count"]
    )
    training_hours_by_seed[run["seed"]] = (
        float(run["training_audit"]["training_seconds"]) / 3600.0
    )

assert overflow_by_seed == {42: 6, 123: 6, 2026: 7}

KD_COMPATIBILITY = {
    "direct_logit_kd": {
        "allowed": True,
        "conditions": [
            "shared vocabulary size 10080",
            "identical PAD/BOS/EOS/UNK token IDs",
            "same decoder-token indexing",
            "fixed temperature and masking contract",
        ],
    },
    "feature_kd": {
        "allowed": True,
        "condition": (
            "Teacher [B,196,256] and S3 [B,49,256] require an explicit "
            "spatial-token alignment or pooling projection."
        ),
    },
    "sequence_kd": {
        "allowed": True,
        "condition": (
            "Use deterministic Beam-3 captions with the locked decoding "
            "policy and Validation-only selection."
        ),
    },
}
write_json(
    CONFIG_DIR / "teacher_c_kd_compatibility_contract.json",
    {
        "contract_version": "teacher_c_kd_compatibility_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "teacher_id": "C",
        "matched_student_id": "S3",
        "siglip_revision": SOURCE_CONTRACT["resolved_commit"],
        "compatibility": KD_COMPATIBILITY,
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
write_json(
    AUDIT_DIR / "phase3l_cell03_metric_protocol.json",
    {
        "audit_version": "phase3l_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "finite_metric_runs": 3,
        "prediction_count_per_seed": 5000,
        "overflow_retries": overflow_by_seed,
        "training_hours": training_hours_by_seed,
        "quality_gates_passed": 3,
        "kd_compatibility": KD_COMPATIBILITY,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03L CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Finite metric runs        : 3 / 3")
print("Prediction count          : 5000 / seed")
print("Quality gates             : 3 / 3 PASSED")
print("Overflow retries          :", overflow_by_seed)
print("Direct logit KD           : ALLOWED WITH LOCKED CONTRACT")
print("Feature KD                : ALLOWED AFTER SPATIAL ALIGNMENT")
print("Sequence KD               : ALLOWED")
print("Test Set status           : SEALED")
print("Next                      : aggregate three seeds")


PHASE 03L CELL 3 STATUS
Status                    : PASSED
Finite metric runs        : 3 / 3
Prediction count          : 5000 / seed
Quality gates             : 3 / 3 PASSED
Overflow retries          : {123: 6, 42: 6, 2026: 7}
Direct logit KD           : ALLOWED WITH LOCKED CONTRACT
Feature KD                : ALLOWED AFTER SPATIAL ALIGNMENT
Sequence KD               : ALLOWED
Test Set status           : SEALED
Next                      : aggregate three seeds


## Cell 4 — Teacher C three-seed aggregate

**Đầu ra dự kiến:** CIDEr-D mean khoảng 1,256165 với sample SD khoảng
0,000554; cả ba run đủ điều kiện KD.


In [4]:
CELL_STARTED_AT = time.perf_counter()
metric_map = {
    "bleu_1": "BLEU-1",
    "bleu_4": "BLEU-4",
    "meteor": "METEOR",
    "rouge_l": "ROUGE-L",
    "cider_d": "CIDEr-D",
}
per_run_rows = []
for run in sorted(run_records, key=lambda item: item["seed"]):
    marker = run["marker"]
    best = marker["best_result"]
    row = {
        "seed": run["seed"],
        "best_epoch": int(best["epoch"]),
        **{
            field: float(best["metrics"][metric])
            for field, metric in metric_map.items()
        },
        "validation_ce": float(best["teacher_forced_ce"]),
        "quality_threshold": float(marker["quality_threshold"]),
        "quality_margin": (
            float(best["metrics"]["CIDEr-D"])
            - float(marker["quality_threshold"])
        ),
        "overflow_retries": int(
            run["training_audit"]["overflow_retry_count"]
        ),
        "training_hours": (
            float(run["training_audit"]["training_seconds"]) / 3600.0
        ),
        "best_checkpoint_sha256": marker["best_checkpoint"]["sha256"],
        "last_checkpoint_sha256": marker["last_checkpoint"]["sha256"],
        "quality_gate_passed": bool(marker["quality_gate_passed"]),
        "qualified_for_kd": bool(marker["qualified_for_kd"]),
        "test_set_status": marker["test_set_status"],
        "test_set_used": bool(marker["test_set_used"]),
    }
    per_run_rows.append(row)

PER_RUN_CSV = RESULT_DIR / "teacher_c_three_seed_per_run.csv"
with PER_RUN_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(per_run_rows[0]))
    writer.writeheader()
    writer.writerows(per_run_rows)

aggregate_metrics = {}
for field in [
    "bleu_1", "bleu_4", "meteor", "rouge_l", "cider_d",
    "validation_ce", "quality_threshold", "quality_margin",
    "training_hours",
]:
    values = [float(row[field]) for row in per_run_rows]
    mean_value = statistics.mean(values)
    sample_sd = statistics.stdev(values)
    aggregate_metrics[field] = {
        "mean": mean_value,
        "sample_sd": sample_sd,
        "cv_percent": (
            100.0 * sample_sd / abs(mean_value)
            if mean_value != 0.0 else None
        ),
        "minimum": min(values),
        "maximum": max(values),
    }

AGGREGATE_JSON = RESULT_DIR / "teacher_c_three_seed_aggregate.json"
aggregate_payload = {
    "schema_version": "teacher_c_three_seed_aggregate_v1",
    "created_at_utc": now_utc(),
    "status": "complete",
    "teacher_id": "C",
    "teacher": "SigLIP ViT-B/16 + shared caption decoder",
    "model_identifier": "google/siglip-base-patch16-224",
    "revision": SOURCE_CONTRACT["resolved_commit"],
    "seeds": [42, 123, 2026],
    "run_count": 3,
    "selection_split": "validation",
    "primary_metric": "CIDEr-D",
    "sample_standard_deviation": True,
    "metrics": aggregate_metrics,
    "all_quality_gates_passed": all(
        row["quality_gate_passed"] for row in per_run_rows
    ),
    "all_qualified_for_kd": all(
        row["qualified_for_kd"] for row in per_run_rows
    ),
    "kd_compatibility": KD_COMPATIBILITY,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(AGGREGATE_JSON, aggregate_payload)
assert math.isclose(
    aggregate_metrics["cider_d"]["mean"],
    1.2561651439784647,
    rel_tol=0.0,
    abs_tol=1e-12,
)
assert math.isclose(
    aggregate_metrics["cider_d"]["sample_sd"],
    0.0005538148433899142,
    rel_tol=0.0,
    abs_tol=1e-12,
)

write_json(
    AUDIT_DIR / "phase3l_cell04_aggregate.json",
    {
        "audit_version": "phase3l_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "per_run_csv": str(PER_RUN_CSV),
        "aggregate_json": str(AGGREGATE_JSON),
        "cider_d": aggregate_metrics["cider_d"],
        "all_quality_gates_passed": True,
        "all_qualified_for_kd": True,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

cider = aggregate_metrics["cider_d"]
print("=" * 72)
print("PHASE 03L CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
for row in per_run_rows:
    print(
        f'Seed {row["seed"]:4d}                  : '
        f'CIDEr-D={row["cider_d"]:.9f} | '
        f'epoch={row["best_epoch"]} | '
        f'margin={row["quality_margin"]:.9f}'
    )
print(
    "CIDEr-D mean ± SD         :",
    f'{cider["mean"]:.9f} ± {cider["sample_sd"]:.9f}',
)
print("CIDEr-D CV                :", f'{cider["cv_percent"]:.6f} %')
print("All seeds qualified       : True")
print("Test Set status           : SEALED")
print("Next                      : checkpoint registry and model card")


PHASE 03L CELL 4 STATUS
Status                    : PASSED
Seed   42                  : CIDEr-D=1.255845496 | epoch=19 | margin=0.178046877
Seed  123                  : CIDEr-D=1.256804634 | epoch=20 | margin=0.179094356
Seed 2026                  : CIDEr-D=1.255845302 | epoch=20 | margin=0.183943434
CIDEr-D mean ± SD         : 1.256165144 ± 0.000553815
CIDEr-D CV                : 0.044088 %
All seeds qualified       : True
Test Set status           : SEALED
Next                      : checkpoint registry and model card


## Cell 5 — Checkpoint registry và model card

**Đầu ra dự kiến:** ba best checkpoint và ba last checkpoint được đăng ký,
không mở checkpoint payload.


In [5]:
CELL_STARTED_AT = time.perf_counter()
registry_entries = []
for run in sorted(run_records, key=lambda item: item["seed"]):
    marker = run["marker"]
    best = marker["best_result"]
    registry_entries.append(
        {
            "teacher_id": "C",
            "seed": run["seed"],
            "model": "google/siglip-base-patch16-224",
            "revision": SOURCE_CONTRACT["resolved_commit"],
            "run_complete_marker": {
                "path": run["marker_path"],
                "sha256": run["marker_sha256"],
            },
            "best_checkpoint": {
                "path": str(resolve_inside(
                    run["run_root"],
                    marker["best_checkpoint"]["relative_path"],
                )),
                "sha256": marker["best_checkpoint"]["sha256"],
                "selection_split": "validation",
                "selection_metric": "CIDEr-D",
                "best_epoch": int(best["epoch"]),
                "cider_d": float(best["metrics"]["CIDEr-D"]),
            },
            "last_checkpoint": {
                "path": str(resolve_inside(
                    run["run_root"],
                    marker["last_checkpoint"]["relative_path"],
                )),
                "sha256": marker["last_checkpoint"]["sha256"],
                "completed_epochs": int(marker["completed_epochs"]),
                "optimizer_steps": int(marker["global_optimizer_steps"]),
            },
            "qualified_for_kd": True,
            "kd_compatibility": KD_COMPATIBILITY,
            "test_set_status": "sealed",
            "test_set_used": False,
        }
    )

REGISTRY_JSON = RESULT_DIR / "teacher_c_checkpoint_registry.json"
write_json(
    REGISTRY_JSON,
    {
        "schema_version": "teacher_c_checkpoint_registry_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "teacher_id": "C",
        "entry_count": 3,
        "entries": registry_entries,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

MODEL_CARD = MODEL_CARD_DIR / "teacher_c_model_card.md"
MODEL_CARD.write_text(
    "\n".join(
        [
            "# Teacher C — SigLIP ViT-B/16",
            "",
            "- Source: `google/siglip-base-patch16-224`",
            f'- Revision: `{SOURCE_CONTRACT["resolved_commit"]}`',
            "- Decoder: locked shared caption decoder",
            "- Dataset: MS COCO 2014 Train + Restval",
            "- Selection: Karpathy Validation only",
            "- Seeds: 42, 123, 2026",
            (
                "- CIDEr-D: "
                f'{aggregate_metrics["cider_d"]["mean"]:.9f} ± '
                f'{aggregate_metrics["cider_d"]["sample_sd"]:.9f} '
                "(sample SD)"
            ),
            (
                "- BLEU-4: "
                f'{aggregate_metrics["bleu_4"]["mean"]:.9f} ± '
                f'{aggregate_metrics["bleu_4"]["sample_sd"]:.9f}'
            ),
            "- Quality gate: 3/3 passed",
            "- Direct logit KD: allowed with locked token contract",
            "- Feature KD: requires 196-to-49 spatial alignment",
            "- Sequence KD: allowed",
            "- Held-out Test: sealed and unused",
            "",
        ]
    ),
    encoding="utf-8",
)

write_json(
    AUDIT_DIR / "phase3l_cell05_registry_model_card.json",
    {
        "audit_version": "phase3l_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "registry_entries": 3,
        "registry": str(REGISTRY_JSON),
        "model_card": str(MODEL_CARD),
        "checkpoint_payload_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03L CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Registry entries          : 3")
print("Best checkpoints          : 3 / 3")
print("Last checkpoints          : 3 / 3")
print("Checkpoint payload read   : False")
print("Registry                  :", REGISTRY_JSON)
print("Model card                :", MODEL_CARD)
print("Test Set status           : SEALED")
print("Next                      : close Phase 3 and authorize Phase 4")


PHASE 03L CELL 5 STATUS
Status                    : PASSED
Registry entries          : 3
Best checkpoints          : 3 / 3
Last checkpoints          : 3 / 3
Checkpoint payload read   : False
Registry                  : /kaggle/working//phase3_teacher_c_three_seed_aggregate/results/teacher_c_checkpoint_registry.json
Model card                : /kaggle/working//phase3_teacher_c_three_seed_aggregate/model_cards/teacher_c_model_card.md
Test Set status           : SEALED
Next                      : close Phase 3 and authorize Phase 4


## Cell 6 — Hoàn tất Teacher pool và khóa Phase 4 entry

**Đầu ra dự kiến:** Teacher A/B/C đều complete, đủ chín Teacher seed runs và
ba cặp matched Student được khóa cho KD reproduction.


In [6]:
CELL_STARTED_AT = time.perf_counter()
assert aggregate_payload["all_quality_gates_passed"] is True
assert aggregate_payload["all_qualified_for_kd"] is True
assert TEACHER_A_COMPLETE["run_count"] == 3
assert TEACHER_A_COMPLETE["quality_gate_passed_seeds"] == 3
assert TEACHER_B_COMPLETE["run_count"] == 3
assert TEACHER_B_COMPLETE["all_quality_gates_passed"] is True

PHASE4_CONTRACT = CONFIG_DIR / "phase4_kd_reproduction_entry_contract.json"
phase4_contract = {
    "contract_version": "phase4_kd_reproduction_entry_v1",
    "created_at_utc": now_utc(),
    "status": "authorized",
    "dataset": "MS COCO 2014 only",
    "split_protocol": "Karpathy",
    "selection_split": "validation",
    "primary_metric": "CIDEr-D",
    "development_seed": 42,
    "matched_pairs": {
        "A_to_S1": {
            "teacher": "CLIP ViT-B/32 + shared decoder",
            "student": "MobileNetV3-Large + shared decoder",
            "allowed_channels": ["logit", "feature", "sequence"],
        },
        "B_to_S2": {
            "teacher": "BLIP Base",
            "student": "MobileViT-S + shared decoder",
            "allowed_channels": ["feature", "sequence"],
            "direct_logit_kd": "prohibited_vocabulary_mismatch",
        },
        "C_to_S3": {
            "teacher": "SigLIP ViT-B/16 + shared decoder",
            "student": "TinyViT-5M + shared decoder",
            "allowed_channels": ["logit", "feature", "sequence"],
            "feature_alignment": "teacher_196_tokens_to_student_49_tokens",
        },
    },
    "phase4_reproduction_policy": {
        "initial_runs": "seed42 matched pairs",
        "student_initialization": "locked Phase 2 baseline initialization",
        "teacher_checkpoint": "best Validation CIDEr-D checkpoint",
        "baseline_comparison": "same Student seed and Validation protocol",
        "test_set_status": "sealed",
        "test_set_used": False,
    },
    "required_phase4a_gates": [
        "verify Teacher and Student checkpoints by SHA-256",
        "lock KD channel compatibility per pair",
        "lock temperature and loss weights",
        "forward/backward finite",
        "one-batch overfit",
        "exact resume",
        "Validation-only checkpoint selection",
        "Held-out Test sealed",
    ],
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(PHASE4_CONTRACT, phase4_contract)

PHASE4_AUTHORIZED = OUTPUT_ROOT / (
    "PHASE4_KD_REPRODUCTION_AUTHORIZED.json"
)
write_json(
    PHASE4_AUTHORIZED,
    {
        "marker_version": "phase4_kd_reproduction_authorized_v1",
        "created_at_utc": now_utc(),
        "status": "authorized",
        "phase3_teacher_families_complete": 3,
        "phase3_teacher_seed_runs_complete": 9,
        "teacher_c_aggregate": {
            "path": str(AGGREGATE_JSON),
            "sha256": sha256_file(AGGREGATE_JSON),
        },
        "teacher_c_registry": {
            "path": str(REGISTRY_JSON),
            "sha256": sha256_file(REGISTRY_JSON),
        },
        "entry_contract": {
            "path": str(PHASE4_CONTRACT),
            "sha256": sha256_file(PHASE4_CONTRACT),
        },
        "next_phase": "Phase 04A KD reproduction entry and compatibility gate",
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

write_json(
    AUDIT_DIR / "phase3l_cell06_phase3_complete.json",
    {
        "audit_version": "phase3l_cell06_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "teacher_families_complete": 3,
        "teacher_seed_runs_complete": 9,
        "phase4_contract": str(PHASE4_CONTRACT),
        "phase4_authorization": str(PHASE4_AUTHORIZED),
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03L CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher families          : 3 / 3 COMPLETE")
print("Teacher seed runs         : 9 / 9 COMPLETE")
print("Matched KD pairs          : 3 / 3 LOCKED")
print("Phase 4 authorization     : READY")
print("KD training started       : False")
print("Test Set status           : SEALED")
print("Next                      : package Phase 03L")


PHASE 03L CELL 6 STATUS
Status                    : PASSED
Teacher families          : 3 / 3 COMPLETE
Teacher seed runs         : 9 / 9 COMPLETE
Matched KD pairs          : 3 / 3 LOCKED
Phase 4 authorization     : READY
KD training started       : False
Test Set status           : SEALED
Next                      : package Phase 03L


## Cell 7 — Package Phase 03L

**Đầu ra bắt buộc:** `PHASE3_TEACHER_POOL_COMPLETE.json`,
`PHASE4_KD_REPRODUCTION_AUTHORIZED.json`, manifest được tự xác minh và Test
vẫn sealed.


In [7]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_CSV = MANIFEST_DIR / "phase3l_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase3l_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase3l_SHA256SUMS.txt"
PHASE3_COMPLETE = OUTPUT_ROOT / "PHASE3_TEACHER_POOL_COMPLETE.json"

excluded = {
    MANIFEST_CSV.resolve(),
    MANIFEST_JSON.resolve(),
    SHA256SUMS.resolve(),
    PHASE3_COMPLETE.resolve(),
}
artifact_paths = sorted(
    [
        path for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path.resolve() not in excluded
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]

with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase3l_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

write_json(
    PHASE3_COMPLETE,
    {
        "marker_version": "phase3_teacher_pool_complete_v1",
        "created_at_utc": now_utc(),
        "status": "complete",
        "teacher_families": ["A", "B", "C"],
        "teacher_family_count": 3,
        "seeds": [42, 123, 2026],
        "teacher_seed_run_count": 9,
        "teacher_c_aggregate": {
            "relative_path": AGGREGATE_JSON.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(AGGREGATE_JSON),
        },
        "teacher_c_checkpoint_registry": {
            "relative_path": REGISTRY_JSON.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(REGISTRY_JSON),
        },
        "phase4_authorization": {
            "relative_path": PHASE4_AUTHORIZED.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(PHASE4_AUTHORIZED),
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
            },
            "json": {
                "relative_path": MANIFEST_JSON.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_JSON),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "manifest_rows": len(manifest_rows),
        "manifest_verification": "all_passed",
        "all_teacher_quality_gates_passed": True,
        "all_teacher_runs_qualified_for_kd": True,
        "phase4_authorized": True,
        "checkpoint_payload_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 03L CELL 7 STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Teacher families          : 3 / 3 COMPLETE")
print("Teacher seed runs         : 9 / 9 COMPLETE")
print("Teacher C quality gates   : 3 / 3 PASSED")
print("Teacher C CIDEr-D mean    :", (
    aggregate_metrics["cider_d"]["mean"]
))
print("Teacher C CIDEr-D SD      :", (
    aggregate_metrics["cider_d"]["sample_sd"]
))
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Phase 4 KD reproduction   : AUTHORIZED")
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Phase 3 marker            :", PHASE3_COMPLETE)
print(
    "Elapsed                   :",
    round(time.perf_counter() - CELL_STARTED_AT, 2),
    "giây",
)
print("Next                      : Save Version, then Phase 04A")


PHASE 03L CELL 7 STATUS
Status                    : COMPLETE
Teacher families          : 3 / 3 COMPLETE
Teacher seed runs         : 9 / 9 COMPLETE
Teacher C quality gates   : 3 / 3 PASSED
Teacher C CIDEr-D mean    : 1.2561651439784647
Teacher C CIDEr-D SD      : 0.0005538148433899142
Manifest rows             : 13
Manifest verification     : ALL PASSED
Phase 4 KD reproduction   : AUTHORIZED
Checkpoint payload read   : False
Test Set status           : SEALED
Test Set used             : False
Phase 3 marker            : /kaggle/working//phase3_teacher_c_three_seed_aggregate/PHASE3_TEACHER_POOL_COMPLETE.json
Elapsed                   : 0.01 giây
Next                      : Save Version, then Phase 04A
